### 1. Installation d'Unsloth

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
import torch
from google.colab import userdata

hf_token = userdata.get('hf_token')

gpu_props = torch.cuda.get_device_properties(0)
vram_gb = round(gpu_props.total_memory / (1024**3), 2)
bf16_ok = torch.cuda.is_bf16_supported()
print(f"🖥️ GPU détecté : {gpu_props.name} ({vram_gb} Go VRAM) | Support bfloat16 natif : {bf16_ok}")

### 2. Chargement 4-bit (`bfloat16`) & Configuration QLoRA Haute Capacité (`r = 32`)

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported

max_seq_length = 2048
dtype = torch.bfloat16 if is_bfloat16_supported() else None  
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    token = hf_token,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 32,              
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,     
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = True,   
    loftq_config = None,
)

### 3. Phase 1 — Continued Pre-Training (CPT) sur les $7\,326$ Textes Camfranglais

Avec `packing = True` et `per_device_train_batch_size = 4` en `bfloat16`, les $7\,326$ commentaires courts sont concaténés en blocs pleins de $2048$ tokens et avalés en quelques minutes par le GPU L4.

In [ ]:
from datasets import load_dataset, Dataset
from trl import SFTConfig, SFTTrainer
import json

CPT_HF_REPO = "Cmr237/camfranglais-corpus-cpt"
cpt_dataset = load_dataset(CPT_HF_REPO, split="train", token=hf_token)
print(f"✅ Corpus CPT téléchargé depuis Hugging Face : {len(cpt_dataset)} textes.")

EOS_TOKEN = tokenizer.eos_token

def format_cpt_texts(examples):
    return {"text": [t.strip() + EOS_TOKEN for t in examples["text"]]}

cpt_dataset = cpt_dataset.map(format_cpt_texts, batched=True, num_proc=2)

model.config.use_cache = False
cpt_trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = cpt_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = True,
    args = SFTConfig(
        per_device_train_batch_size = 4,   
        gradient_accumulation_steps = 2,   
        warmup_ratio = 0.05,
        num_train_epochs = 1,
        learning_rate = 1e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),    
        save_strategy = "no",
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        dataloader_num_workers = 2,
        seed = 3407,
        output_dir = "outputs_cpt",
        report_to = "none",
    ),
)

cpt_trainer_stats = cpt_trainer.train()

### 4. Phase 2 — Supervised Fine-Tuning (SFT Conversationnel sur les $667$ Dialogues)

On passe maintenant aux dialogues `Cmr237/camfranglais-corpus-sft` avec :
- Le **Chat Template natif de Mistral** (`[INST] ... [/INST]`).
- Le masquage de la *loss* sur l'instruction via `train_on_responses_only`.
- `per_device_train_batch_size = 4`, `bf16 = True` et `learning_rate = 5e-5`.

In [ ]:
from unsloth.chat_templates import get_chat_template, train_on_responses_only

tokenizer = get_chat_template(tokenizer, chat_template="mistral")
SYSTEM_PROMPT = "Tu réponds uniquement en camfranglais"

SFT_HF_REPO = "Cmr237/camfranglais-corpus-sft"
sft_dataset_dict = load_dataset(SFT_HF_REPO, token=hf_token)
sft_train_dataset = sft_dataset_dict["train"]
sft_test_dataset = sft_dataset_dict["test"]
print(f"✅ Dataset SFT téléchargé : {len(sft_train_dataset)} train | {len(sft_test_dataset)} test.")

def formatted_conversations(examples):
    texts = []
    for convo in examples["conversations"]:
        full_convo = [{"role": "system", "content": SYSTEM_PROMPT}] + convo
        texts.append(tokenizer.apply_chat_template(full_convo, tokenize=False, add_generation_prompt=False))
    return {"text": texts}

sft_train_dataset = sft_train_dataset.map(formatted_conversations, batched=True)
sft_test_dataset = sft_test_dataset.map(formatted_conversations, batched=True)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = sft_train_dataset,
    eval_dataset = sft_test_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = 4,   
        per_device_eval_batch_size = 4,
        gradient_accumulation_steps = 2,   
        warmup_ratio = 0.05,
        num_train_epochs = 2,
        learning_rate = 5e-5,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),    
        save_strategy = "epoch",
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        dataloader_num_workers = 2,
        seed = 3407,
        output_dir = "outputs_sft",
        report_to = "none",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "[INST]",
    response_part = "[/INST]",
)

trainer_stats = trainer.train()

### 5. Évaluation sur les $74$ Dialogues de `sft_test` & Génération de `test_predictions.json` (pour l'Étape 5 en Local)

In [ ]:
import math

eval_metrics = trainer.evaluate()
eval_loss = eval_metrics.get("eval_loss", 0.0)
perplexity = math.exp(eval_loss) if eval_loss < 20 else float("inf")
print(f"📊 Eval Loss sur sft_test = {eval_loss:.4f} | Perplexité (PPL) = {perplexity:.2f}")

FastLanguageModel.for_inference(model)

test_predictions = []
for item in sft_test_dataset:
    convo = item["conversations"]
    user_q = convo[0]["content"]
    human_ref = convo[1]["content"]

    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_q}
    ]
    inputs = tokenizer.apply_chat_template(
        msgs,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=128,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    gen_ids = outputs[0][inputs.shape[1]:]
    model_ans = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    test_predictions.append({
        "user": user_q,
        "human_reference": human_ref,
        "model_response": model_ans
    })

with open("test_predictions.json", "w", encoding="utf-8") as f:
    json.dump({
        "eval_loss": eval_loss,
        "perplexity": perplexity,
        "predictions": test_predictions
    }, f, indent=2, ensure_ascii=False)

print(f"💾 {len(test_predictions)} réponses générées sauvegardées dans test_predictions.json !")

### 6. Inférence Interactive (`TextStreamer`)

In [ ]:
FastLanguageModel.for_inference(model)
from transformers import TextStreamer

messages = [{"role": "system", "content": SYSTEM_PROMPT}]
while True:
    user_input = str(input("user: \n"))
    if user_input.lower() == "exit":
        break
    messages.append({"role": "user", "content": user_input})
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize = True,
        add_generation_prompt = True,
        return_tensors = "pt",
    ).to("cuda")

    text_streamer = TextStreamer(tokenizer, skip_prompt = True, skip_special_tokens = True)
    print("Assistant: ", end="", flush=True)
    outputs = model.generate(
        input_ids = inputs,
        streamer = text_streamer,
        max_new_tokens = 128,
        pad_token_id = tokenizer.eos_token_id
    )
    prompt_length = inputs.shape[1]
    response_ids = outputs[0][prompt_length:]
    response_text = tokenizer.decode(response_ids, skip_special_tokens=True).strip()
    messages.append({"role": "assistant", "content": response_text})

### 7. Sauvegarde des Adaptateurs LoRA & Export GGUF (`q4_k_m`) vers Hugging Face

In [ ]:
model.push_to_hub("Cmr237/Camfranglais", token = hf_token)
tokenizer.push_to_hub("Cmr237/Camfranglais", token = hf_token)

model.push_to_hub_gguf(
    "Cmr237/Camfranglais",
    tokenizer,
    quantization_method = "q4_k_m",
    token = hf_token,
)